# ESM record ID lists of the ESHM20 regional datasets

The ESHM20 subduction in-slab, subduction interface and Vrancea datasets were provided by
Graeme Weatherill (GFZ) as JSON files. Every record in them is an ESM flatfile record, so each
dataset is fully described by the ESM identifiers of its records. This notebook writes those
identifiers, one CSV per dataset, to `data_raw/eshm20_trt_record_ids/`
(`cfg["raw_data"]["trt_record_ids"]`). nb `002-im_correlations` rebuilds the regional
flatfiles from these lists and the ESM flatfile, so it does not need the JSON files.

Each list holds the four columns `manage_flatfiles` matches records on (`event_id`,
`network_code`, `station_code`, `sensor_depth_m`), converted exactly as
`extract_flatfile_for_dataset` converts them: the codes as strings, the sensor depth as float64.
One row per record, no duplicates, sorted on the four columns.

In [1]:
import json
import pandas as pd

from phd_project.config.config import load_config
import phd_project.scripts.WP1_ground_motion_set.manage_flatfiles as mf

cfg = load_config()

In [2]:
# The GFZ datasets and the region tag nb 002 uses for each
gfz_folder = cfg["raw_data"]["root"] / "eshm20_subduction_vrancea_datasets"
gfz_files = {
    "sslab": "hellenic_inslab_db.json",
    "sinter": "hellenic_interface_db.json",
    "vran": "vrancea_db.json",
    }

out_folder = cfg["raw_data"]["trt_record_ids"]
out_folder.mkdir(parents=True, exist_ok=True)

In [3]:
# Write one ID list per dataset, then read it back the way nb 002 does and check it is
# unchanged (codes such as station "0703" keep their leading zero, missing depths stay NaN)
for tag, file_name in gfz_files.items():
    with open(gfz_folder / file_name, "r") as file:
        db = json.load(file)

    ids = mf.record_ids_from_dataset(db)
    fp = out_folder / f"{tag}_esm_record_ids.csv"
    ids.to_csv(fp, sep=",", index=False)

    pd.testing.assert_frame_equal(mf.load_esm_record_ids(fp), ids, check_exact=True)
    print(f"{tag:<7}: {len(db['records']):>5} records in {file_name:<27} -> "
          f"{len(ids):>5} record IDs in {fp.name}")

sslab  :   704 records in hellenic_inslab_db.json     ->   704 record IDs in sslab_esm_record_ids.csv
sinter :   681 records in hellenic_interface_db.json  ->   681 record IDs in sinter_esm_record_ids.csv


vran   :   938 records in vrancea_db.json             ->   938 record IDs in vran_esm_record_ids.csv
